### Import Library

In [1]:
import pandas as pd
import re
from langdetect import detect, DetectorFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

### Load Dataset

In [16]:
df = pd.read_csv('../../data/raw/dataset.csv')

print('Jumlah data awal :', len(df))
print('Kolom            :', df.columns.tolist())
print()
df.info()

Jumlah data awal : 62614
Kolom            : ['author_tweet', 'username', 'name', 'created_at', 'text', 'likes', 'retweets']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62614 entries, 0 to 62613
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   author_tweet  62614 non-null  object
 1   username      62614 non-null  object
 2   name          62613 non-null  object
 3   created_at    62614 non-null  object
 4   text          62614 non-null  object
 5   likes         62614 non-null  int64 
 6   retweets      62614 non-null  int64 
dtypes: int64(2), object(5)
memory usage: 3.3+ MB


### Preprocessing

Tahapan ini diterapkan untuk RoBERTa, IndoBERTa, dan InSet Lexicon :
1. Hapus kolom yang tidak digunakan
2. Hapus missing value
3. Hapus duplikat
4. Case folding
5. Cleaning (URL, mention, hashtag, simbol, angka)
6. Hapus baris kosong & duplikat setelah cleaning
7. Normalisasi slang
8. Stopword removal (dengan mempertahankan kata negasi)
9. Stemming
10. Tokenisasi

In [17]:
# Hapus kolom yang tidak dipakai
df = df.drop(['author_tweet', 'username', 'name', 'created_at', 'likes', 'retweets'], axis=1)

# Hapus missing value & duplikat
df = df.dropna(subset=['text'])
df = df.drop_duplicates(subset='text')
df = df.reset_index(drop=True)
print("Jumlah data awal:", len(df))

df.head(10)

Jumlah data awal: 54964


,text
0,@jokowi @prabowo Adili Jokowi
1,@jokowi @prabowo Semoga pak prabowo ngga kayak...
2,@jokowi @prabowo Siap siap ya MULYONO semua me...
3,@jokowi @prabowo Kenapa pilih Poto yg ini? Bia...
4,@jokowi @prabowo Loe bayar nih utang https://t...
5,@jokowi @prabowo Ijazah lu mana wi..? https://...
6,@jokowi @prabowo Jangan sampai terjadi lagi In...
7,@jokowi @prabowo Sir koq lebih tertarik dengan...
8,@jokowi @prabowo https://t.co/wMNuUnxjzr
9,@jokowi @prabowo Warisanmu nyusahin presiden s...


In [19]:
# Cleaning text
def original_text(text):
    text = text.lower()                           # Case folding
    text = re.sub(r'http\S+', '', text)           # Hapus URL
    text = re.sub(r'@\w+', '', text)              # Hapus mention
    text = re.sub(r'#\w+', '', text)              # Hapus hashtag
    text = re.sub(r'[^a-zA-Z\s]', '', text)       # Hapus simbol & angka
    text = re.sub(r'\s+', ' ', text).strip()      # Hapus spasi berlebih
    return text

df['original_text'] = df['text'].apply(original_text)

# Hapus baris kosong & duplikat setelah cleaning
df = df[df['original_text'] != '']
df = df.drop_duplicates(subset='original_text')
df = df.reset_index(drop=True)
print("Jumlah data setelah cleaning:", len(df))

df.head(10)

Jumlah data setelah cleaning: 48546


,text,original_text
0,@jokowi @prabowo Adili Jokowi,adili jokowi
1,@jokowi @prabowo Semoga pak prabowo ngga kayak...,semoga pak prabowo ngga kayak gini ya pak
2,@jokowi @prabowo Siap siap ya MULYONO semua me...,siap siap ya mulyono semua memanggilmu
3,@jokowi @prabowo Kenapa pilih Poto yg ini? Bia...,kenapa pilih poto yg ini biar masih berkesan s...
4,@jokowi @prabowo Loe bayar nih utang https://t...,loe bayar nih utang
5,@jokowi @prabowo Ijazah lu mana wi..? https://...,ijazah lu mana wi
6,@jokowi @prabowo Jangan sampai terjadi lagi In...,jangan sampai terjadi lagi indonesia pernah di...
7,@jokowi @prabowo Sir koq lebih tertarik dengan...,sir koq lebih tertarik dengan pinjaman yg bung...
8,@jokowi @prabowo Warisanmu nyusahin presiden s...,warisanmu nyusahin presiden skg pak
9,@jokowi @prabowo Hallo .... Pak @prabowo 👇👇 ht...,hallo pak


In [21]:
# Normalized
kamus_slang = pd.read_excel('../../data/kamus/kamus_slang.xlsx')

# Ubah jadi dictionary {slang: formal}
slang_dict = dict(zip(kamus_slang['slang'], kamus_slang['formal']))

# Fungsi normalisasi slang
def normalize_slang(text):
    words = text.split()
    normalized = [slang_dict.get(word, word) for word in words]
    return ' '.join(normalized)

df['normalized'] = df['original_text'].apply(normalize_slang)

# Cek hasil
print("Contoh hasil normalisasi:")
for i in range(3):
    print(f"Sebelum : {df['original_text'][i]}")
    print(f"Sesudah : {df['normalized'][i]}")
    print()

df.head(5)   

Contoh hasil normalisasi:
Sebelum : adili jokowi
Sesudah : adili jokowi

Sebelum : semoga pak prabowo ngga kayak gini ya pak
Sesudah : semoga pak prabowo tidak kayak begini ya pak

Sebelum : siap siap ya mulyono semua memanggilmu
Sesudah : siap siap ya mulyono semua memanggilmu



,text,original_text,normalized
0,@jokowi @prabowo Adili Jokowi,adili jokowi,adili jokowi
1,@jokowi @prabowo Semoga pak prabowo ngga kayak...,semoga pak prabowo ngga kayak gini ya pak,semoga pak prabowo tidak kayak begini ya pak
2,@jokowi @prabowo Siap siap ya MULYONO semua me...,siap siap ya mulyono semua memanggilmu,siap siap ya mulyono semua memanggilmu
3,@jokowi @prabowo Kenapa pilih Poto yg ini? Bia...,kenapa pilih poto yg ini biar masih berkesan s...,kenapa pilih foto yang ini biar masih berkesan...
4,@jokowi @prabowo Loe bayar nih utang https://t...,loe bayar nih utang,lo bayar nih utang


In [22]:
# Stopword removal

factory = StopWordRemoverFactory()
stopwords = set(factory.get_stop_words())

negation_words = {'tidak','bukan','jangan','belum','tanpa','kurang'}

# remove dari stopword
stopwords = stopwords - negation_words

def remove_stopwords(text):
    words = text.split()
    filtered = [w for w in words if w not in stopwords]
    return ' '.join(filtered)

df['normalized'] = df['normalized'].apply(remove_stopwords)

df.rename(columns={'normalized':'stopword'}, inplace=True)

df.head(10)

,text,original_text,stopword
0,@jokowi @prabowo Adili Jokowi,adili jokowi,adili jokowi
1,@jokowi @prabowo Semoga pak prabowo ngga kayak...,semoga pak prabowo ngga kayak gini ya pak,semoga pak prabowo tidak kayak begini pak
2,@jokowi @prabowo Siap siap ya MULYONO semua me...,siap siap ya mulyono semua memanggilmu,siap siap mulyono semua memanggilmu
3,@jokowi @prabowo Kenapa pilih Poto yg ini? Bia...,kenapa pilih poto yg ini biar masih berkesan s...,pilih foto biar berkesan siapa tuannya
4,@jokowi @prabowo Loe bayar nih utang https://t...,loe bayar nih utang,lo bayar nih utang
5,@jokowi @prabowo Ijazah lu mana wi..? https://...,ijazah lu mana wi,ijazah kamu mana wi
6,@jokowi @prabowo Jangan sampai terjadi lagi In...,jangan sampai terjadi lagi indonesia pernah di...,jangan terjadi indonesia pernah ditipu satu ke...
7,@jokowi @prabowo Sir koq lebih tertarik dengan...,sir koq lebih tertarik dengan pinjaman yg bung...,sir kok lebih tertarik pinjaman bunganya lebih...
8,@jokowi @prabowo Warisanmu nyusahin presiden s...,warisanmu nyusahin presiden skg pak,warisanmu menyusahkan presiden sekarang pak
9,@jokowi @prabowo Hallo .... Pak @prabowo 👇👇 ht...,hallo pak,halo pak


In [23]:
# stemming

factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stemming(text):
    return stemmer.stem(text)

df['stopword'] = df['stopword'].apply(stemming)

df.rename(columns={'stopword':'stemming'}, inplace=True)

df.head(10)

,text,original_text,stemming
0,@jokowi @prabowo Adili Jokowi,adili jokowi,adil jokowi
1,@jokowi @prabowo Semoga pak prabowo ngga kayak...,semoga pak prabowo ngga kayak gini ya pak,moga pak prabowo tidak kayak begini pak
2,@jokowi @prabowo Siap siap ya MULYONO semua me...,siap siap ya mulyono semua memanggilmu,siap siap mulyono semua panggil
3,@jokowi @prabowo Kenapa pilih Poto yg ini? Bia...,kenapa pilih poto yg ini biar masih berkesan s...,pilih foto biar kesan siapa tuan
4,@jokowi @prabowo Loe bayar nih utang https://t...,loe bayar nih utang,lo bayar nih utang
5,@jokowi @prabowo Ijazah lu mana wi..? https://...,ijazah lu mana wi,ijazah kamu mana wi
6,@jokowi @prabowo Jangan sampai terjadi lagi In...,jangan sampai terjadi lagi indonesia pernah di...,jangan jadi indonesia pernah tipu satu keluarg...
7,@jokowi @prabowo Sir koq lebih tertarik dengan...,sir koq lebih tertarik dengan pinjaman yg bung...,sir kok lebih tarik pinjam bunga lebih tinggi ...
8,@jokowi @prabowo Warisanmu nyusahin presiden s...,warisanmu nyusahin presiden skg pak,waris susah presiden sekarang pak
9,@jokowi @prabowo Hallo .... Pak @prabowo 👇👇 ht...,hallo pak,halo pak


In [24]:
# tokenize

def tokenize_safe(text):
    if not isinstance(text, str):
        return []
    return text.split()

df['stemming'] = df['stemming'].apply(tokenize_safe)

df.rename(columns={'stemming':'clean_text'}, inplace=True)

df.head(10)

,text,original_text,clean_text
0,@jokowi @prabowo Adili Jokowi,adili jokowi,"[adil, jokowi]"
1,@jokowi @prabowo Semoga pak prabowo ngga kayak...,semoga pak prabowo ngga kayak gini ya pak,"[moga, pak, prabowo, tidak, kayak, begini, pak]"
2,@jokowi @prabowo Siap siap ya MULYONO semua me...,siap siap ya mulyono semua memanggilmu,"[siap, siap, mulyono, semua, panggil]"
3,@jokowi @prabowo Kenapa pilih Poto yg ini? Bia...,kenapa pilih poto yg ini biar masih berkesan s...,"[pilih, foto, biar, kesan, siapa, tuan]"
4,@jokowi @prabowo Loe bayar nih utang https://t...,loe bayar nih utang,"[lo, bayar, nih, utang]"
5,@jokowi @prabowo Ijazah lu mana wi..? https://...,ijazah lu mana wi,"[ijazah, kamu, mana, wi]"
6,@jokowi @prabowo Jangan sampai terjadi lagi In...,jangan sampai terjadi lagi indonesia pernah di...,"[jangan, jadi, indonesia, pernah, tipu, satu, ..."
7,@jokowi @prabowo Sir koq lebih tertarik dengan...,sir koq lebih tertarik dengan pinjaman yg bung...,"[sir, kok, lebih, tarik, pinjam, bunga, lebih,..."
8,@jokowi @prabowo Warisanmu nyusahin presiden s...,warisanmu nyusahin presiden skg pak,"[waris, susah, presiden, sekarang, pak]"
9,@jokowi @prabowo Hallo .... Pak @prabowo 👇👇 ht...,hallo pak,"[halo, pak]"


In [26]:
df['clean_text'] = df['clean_text'].apply(lambda x: ' '.join(x))

In [27]:
df.to_csv('../../data/preprocessing/hasil_preprocessing.csv', index=False)